In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)

Project root added: /home/junix/marketing-content-verfication


In [6]:
from src.ingestion.load_products import load_products

df = load_products(
    "../data/raw/products.csv"
)

[INFO] Loaded 100 products.


In [7]:
from src.ingestion.load_products import load_products

from src.ingestion.preprocess import (
    preprocess_dataframe
)

df = load_products(
    "../data/raw/products.csv"
)

df = preprocess_dataframe(df)

[INFO] Loaded 100 products.


In [8]:
# from pathlib import Path

# print(Path.cwd())

In [9]:
# import os

# print(os.listdir())

In [10]:
print(df.columns.tolist())

['product_id', 'product_name', 'category', 'brand', 'source', 'price_inr', 'product_description', 'specifications_and_warranty', 'source_url']


In [11]:
import importlib

import src.ingestion.chunking as chunking

importlib.reload(chunking)

<module 'src.ingestion.chunking' from '/home/junix/marketing-content-verfication/src/ingestion/chunking.py'>

In [12]:
from src.ingestion.chunking import (
    create_chunks
)

In [13]:
product_chunks = create_chunks(
    df,
    strategy="product"
)

print(
    "Number of chunks:",
    len(product_chunks)
)

print(
    product_chunks[0]
)

[INFO] Created 100 chunks using 'product' strategy.
Number of chunks: 100
{'product_id': 'Elec-01', 'product_name': 'boAt Airdopes 141 Gen 2 TWS Earbuds', 'category': 'Electronics', 'chunk_type': 'product', 'text': 'Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds\n            Brand: boAt\n            Category: Electronics\n            Price: ₹999\n\n            Description:\n            True wireless earbuds with 48-hour total playback, 4-mic ENx tech for crystal-clear calls, Beast Mode low-latency gaming, and IPX4 sweat resistance.\n\n            Specifications & Warranty:\n            Drivers: 6mm | BT: v5.4 | Playback: 48 hrs (buds+case) | Fast Charge: 10 min = 180 min | IPX4 | Warranty: 1 Year'}


In [14]:
attribute_chunks = create_chunks(
    df,
    strategy="attribute"
)

print(
    "Number of chunks:",
    len(attribute_chunks)
)

print(
    attribute_chunks[0]
)

[INFO] Created 607 chunks using 'attribute' strategy.
Number of chunks: 607
{'product_id': 'Elec-01', 'product_name': 'boAt Airdopes 141 Gen 2 TWS Earbuds', 'category': 'Electronics', 'chunk_type': 'attribute', 'text': 'Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds\n                Brand: boAt\n                Category: Electronics\n\n                Drivers: 6mm'}


In [15]:
print(
    df["specifications_and_warranty"].iloc[0]
)

Drivers: 6mm | BT: v5.4 | Playback: 48 hrs (buds+case) | Fast Charge: 10 min = 180 min | IPX4 | Warranty: 1 Year


In [16]:
print("Product Chunks:", len(product_chunks))
print("Attribute Chunks:", len(attribute_chunks))

Product Chunks: 100
Attribute Chunks: 607


** Experiment 1 complete **

In [17]:
from src.embeddings.embed_products import (
    embed_documents
)

In [18]:
import sys

print(sys.executable)

/home/junix/marketing-content-verfication/venv/bin/python


**for bge-small-en-v1.5**

In [19]:
embeddings = embed_documents(
    attribute_chunks,
    "BAAI/bge-small-en-v1.5"
)

[INFO] Loading model: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

[INFO] Generated embeddings for 607 chunks.


In [20]:
print(embeddings.shape)

(607, 384)


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
query = embeddings[0]

candidate = embeddings[1]

score = cosine_similarity(
    [query],
    [candidate]
)

print(score)

[[0.9470808]]


In [23]:
print(
    "Embedding Shape:",
    embeddings.shape
)

Embedding Shape: (607, 384)


In [24]:
from src.vectordb.create_faiss import (
    create_index,
    save_index,
    load_index
)

from src.vectordb.search_faiss import (
    search_index
)

In [25]:
index = create_index(
    embeddings
)

[INFO] Added 607 vectors to FAISS.


In [26]:
save_index(
    index,
    "../models/faiss/products.index"
)

[INFO] Saved index: ../models/faiss/products.index


In [27]:
index = load_index(
    "../models/faiss/products.index"
)

[INFO] Loaded index: ../models/faiss/products.index


In [28]:
query_embedding = embeddings[0].reshape(1, -1)

distances, indices = search_index(
    index,
    query_embedding,
    k=5
)

print(indices)
print(distances)

[[0 1 4 2 5]]
[[1.         0.9470807  0.93962324 0.9100426  0.8959179 ]]


In [29]:
for idx in indices[0]:
    print(attribute_chunks[idx]["text"])
    print("-" * 50)

Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                Drivers: 6mm
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                BT: v5.4
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                IPX4
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                Playback: 48 hrs (buds+case)
--------------------------------------------------
Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
                Brand: boAt
                Category: Electronics

                Warranty: 1 Year
--------------------------------------------------
